# Run training sufficient for inference
This notebook does not introduce new functionality. It uses the extracted, refactored python modules to run extensive training that will be sufficient for inference.

Alternatively, the same training can be accomplished by using the CLI scripts (`nanollm-train`, `nanollm-resume`), which use the same underlying python modules.

## Load untrained model

In [ ]:
from src.config import ModelConfig
from src.model.model import NanoLLM

model_config = ModelConfig()
fresh_model = NanoLLM(model_config) 

# initialize structure to capture loss history across phases
all_losses = []

## Run initial training phase

In [ ]:
from src.checkpoint import default_checkpoint_path
from src.config import TokenizerConfig, TrainingConfig 
from src.paths import DEFAULT_DATA_FILE
from src.training.runner import Runner

TRAINING_EPOCHS = 25 # will be used for all phases                                                                    
training_config = TrainingConfig(epochs=TRAINING_EPOCHS)

# phase 1 (persists the initial checkpoint)
runner = Runner(
    model=fresh_model,
    tokenizer_config=TokenizerConfig(),
    data_source=DEFAULT_DATA_FILE,
    training_config=training_config,
    checkpoint_destination=default_checkpoint_path(),
)

# returns metrics history logged by epoch
history = runner.run()
print(f"Phase 1: {history}")

# append for visualization over all epochs
all_losses.extend(history.train_loss)

## Resume training in multiple phases

In [ ]:
from src.checkpoint import get_latest_checkpoint, restore_from_checkpoint
from src.training.schema import ResumeContext

# run subsequent phases - each loading the most recently persisted checkpoint and persisting a new checkpoint
PHASES = 3
for phase in range(PHASES): 
    most_recent_checkpoint = get_latest_checkpoint()
    loaded_model, tokenizer_config = restore_from_checkpoint(most_recent_checkpoint)
    resume_ctx = ResumeContext.from_checkpoint(most_recent_checkpoint)

    runner = Runner(
        model=loaded_model,
        tokenizer_config=tokenizer_config,
        data_source=DEFAULT_DATA_FILE,
        training_config=training_config,
        checkpoint_destination=default_checkpoint_path(), # execute on each run to generate new timestamped directory name
        resume_from=resume_ctx,
    )

    history = runner.run()
    print(f"Phase {phase + 2}: {history}")

    # append for visualization over all epochs
    all_losses.extend(history.train_loss)

## Visualize Training

In [ ]:
import pprint
from src.checkpoint import load_metadata

# load cumulative training metrics
most_recent_checkpoint = get_latest_checkpoint()
pprint.pp(load_metadata(most_recent_checkpoint))                                                                                     

In [ ]:
import matplotlib.pyplot as plt

plt.plot(all_losses)
plt.title('Training Loss (All Epochs)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()